# Headless Reduction Pipeline

The canonical headless reduction workflow in `ssrl_xrd_tools.reduction`:

1. Wrap your data in a **`Scan`** of **`Frame`** objects (one per
   detector image, with optional lazy loading).
2. Describe what to do in a **`ReductionPlan`** (1D / 2D integration
   settings, optional GI mode, mask, threshold).
3. Pick a **sink** — `MemorySink` for in-notebook follow-up,
   `NexusSink` for persistent batch processing, or your own custom
   sink (e.g. a Qt signal sink for GUI streaming).
4. Run **`run_reduction(plan, scan, sink, *, chunk_size, progress_cb,
   cancel_token)`** — same call works in a notebook, a CLI, or
   xdart's wrangler threads.

This is the API the xdart wranglers were rewritten on top of —
demonstrating it here in a notebook gives you the same canonical
path that the GUI uses, without the GUI.


## Imports

In [ ]:
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from natsort import natsorted

from ssrl_xrd_tools.io import load_mask, read_image
from ssrl_xrd_tools.integrate import load_poni
from ssrl_xrd_tools.reduction import (
    CancelToken,
    Frame,
    GIMode,
    Integration1DPlan,
    Integration2DPlan,
    MaskSpec,
    MemorySink,
    NexusSink,
    ReductionPlan,
    ReductionProgress,
    Scan,
    run_reduction,
)

## ✏️ Configuration

**Edit the cell below**, then run the rest of the notebook top-to-bottom.

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║                  EDIT THIS CELL — paths + parameters                  ║
# ╚══════════════════════════════════════════════════════════════════════╝

# ── REQUIRED ───────────────────────────────────────────────────────────
data_dir   = Path('~/data/my_scan').expanduser()    # ← REPLACE
poni_file  = data_dir / 'calibration.poni'           # ← REPLACE
image_glob = '*.tif'                                  # pattern inside data_dir

# ── OPTIONAL paths ─────────────────────────────────────────────────────
mask_file  = data_dir / 'mask.edf'           # set to None if no mask
out_nxs    = data_dir / 'reduced.nxs'

# ── Integration tuning ─────────────────────────────────────────────────
npt_1d                   = 1000
npt_2d_rad, npt_2d_azim  = 1000, 360
unit                     = 'q_A^-1'
method                   = 'csr'

# ── Grazing incidence (optional) ──────────────────────────────────────
# Set to None for transmission geometry.  Uncomment + set for GI.
gi: GIMode | None = None
# gi = GIMode(incident_angle=0.5)

# ── Monitor normalisation (optional) ──────────────────────────────────
# Key in each Frame.metadata that holds the i0/monitor reading.
monitor_key: str | None = None       # e.g. 'i0'

### Validate the config

In [ ]:
assert data_dir.is_dir(), f'data_dir not found: {data_dir}'
assert poni_file.is_file(), f'poni_file not found: {poni_file}'
if mask_file is not None and not mask_file.exists():
    mask_file = None                 # silently disable if missing
_images_preview = sorted(data_dir.glob(image_glob))
assert _images_preview, f'No images match {image_glob!r} in {data_dir}'
print(f'OK — {len(_images_preview)} image(s) found, PONI present.')

## Build the `Scan` + `Frame` list

In [ ]:
poni = load_poni(poni_file)
mask = load_mask(mask_file) if mask_file.exists() else None

# Lazy loading: pass source_path instead of image=, and Frame.load_image()
# reads the file on demand inside run_reduction.  Use clear_frame_images=
# below to release each frame after it's been written to the sink.
image_files = natsorted(data_dir.glob(image_glob))
print(f'Found {len(image_files)} images')

frames = [
    Frame(
        index=i,
        source_path=path,
        # metadata can carry per-frame counters that monitor_key picks up
        metadata={'i0': 1.0},   # replace with your actual i0 readouts
    )
    for i, path in enumerate(image_files)
]

scan = Scan(
    name='my_scan',
    frames=frames,
    poni=poni,
    output_path=out_nxs,
)
print(f'Built Scan with {len(scan)} frames')

## Compose a `ReductionPlan`

The plan captures *what* the reduction does — integration settings,
mask, threshold, GI mode.  Execution-policy knobs like `chunk_size` /
`clear_frame_images` are arguments to `run_reduction`, not part of the
plan, so the same plan can be re-run on different scans with
different chunking.

In [ ]:
plan = ReductionPlan(
    integration_1d=Integration1DPlan(
        npt=npt_1d,
        unit=unit,
        method=method,
        monitor_key=monitor_key,
    ),
    integration_2d=Integration2DPlan(
        npt_rad=npt_2d_rad,
        npt_azim=npt_2d_azim,
        unit=unit,
        method=method,
        monitor_key=monitor_key,
    ),
    # MaskSpec lets you pass a flat-index mask without knowing the
    # detector shape upfront; the executor resolves it when the first
    # frame is loaded.  Pass a plain 2D bool array to skip the wrapper.
    mask=MaskSpec(mask) if mask is not None else None,
    threshold_min=None,                # e.g. 0 to clip negative dark
    threshold_max=None,                # e.g. 1e8 to clip hot pixels
    gi=gi,                             # None for transmission, GIMode(...) for GI
)
print(plan)

## Run with the in-memory sink

`MemorySink` collects every frame's `FrameReduction` into a dict —
handy in notebooks where you want to inspect / plot the results
immediately.

In [ ]:
sink = MemorySink()
progress_events: list[ReductionProgress] = []

result = run_reduction(
    plan, scan, sink,
    chunk_size=8,                    # tune to your detector + memory
    clear_frame_images=True,         # release each frame's image after sink.write
    progress_cb=progress_events.append,
)

print(f'Processed {result.n_processed}/{len(scan)} frames')
print(f'Last 3 progress events: {progress_events[-3:]}')
first = sink.frames[0]
print(f'Frame 0: 1D shape {first.result_1d.intensity.shape}, '
      f'2D shape {first.result_2d.intensity.shape}')

## Re-run with a `NexusSink` for persistent batch output

In [ ]:
# Same plan + scan, different sink.  The .nxs file is written
# frame-by-frame; flush_every=16 means the OS flushes after every
# 16 frames so a crashed run is mostly recoverable.
nx_sink = NexusSink(out_nxs, overwrite=True, flush_every=16)

result = run_reduction(
    plan, scan, nx_sink,
    chunk_size=8,
    clear_frame_images=True,
)
print(f'Wrote {result.n_processed} frames to {result.output_path}')

## Cancellation

Any consumer can flip the cancel token to stop the loop at the next
frame boundary.  Useful for GUI Stop buttons / Ctrl-C in notebooks.

In [ ]:
token = CancelToken()

# Simulate a cancel after 3 frames have been processed.
def _cancel_at_3(event: ReductionProgress) -> None:
    if event.stage == 'write' and event.completed >= 3:
        token.cancel()

partial = run_reduction(
    plan, scan, MemorySink(),
    chunk_size=4,
    progress_cb=_cancel_at_3,
    cancel_token=token,
)
print(f'Cancelled? {partial.cancelled}; '
      f'processed {partial.n_processed} of {len(scan)}')

## Plot the first frame's results

In [ ]:
r1 = sink.frames[0].result_1d
r2 = sink.frames[0].result_2d

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
ax1.plot(r1.radial, r1.intensity)
ax1.set_xlabel(f'q ({r1.unit})')
ax1.set_ylabel('Intensity')

im = ax2.pcolormesh(r2.radial, r2.azimuthal, r2.intensity.T,
                    shading='auto', cmap='viridis')
ax2.set_xlabel(f'q ({r2.unit})')
ax2.set_ylabel('χ (deg)')
plt.colorbar(im, ax=ax2)
plt.tight_layout()
plt.show()

---

### Notes

- **GI workflow**: set `gi=GIMode(incident_angle=...)` on the plan;
  the executor builds a `FiberIntegrator` from `scan.poni` + the
  GIMode parameters and routes through `integrate_gi_1d` /
  `integrate_gi_2d`.
- **Custom sinks**: any class with `begin(scan, plan)` /
  `write(frame, reduction)` / `finish(result)` methods is a valid
  `ReductionSink` (it's a `typing.Protocol`).  xdart adds its own
  Qt-signal sink on top of this; you can do the same with a plain
  Python collector, a Tiled writer, a CSV row appender, etc.
- **Same code in xdart**: the GUI's wrangler threads build a
  `ReductionPlan` from sphere settings and call `run_reduction` —
  the same workflow you just ran.  See the `keep_xdart_thin.md`
  memory note for the design rationale.
